# Van verzoek naar resultaat

Een terugblik op het Feature-werk van de afgelopen anderhalf jaar: wat is afgerond, hoe lang stond werk te wachten en hoe lang liep het daarna door?

De cijfers maken onderscheid tussen features voor de HU-domeinteams en D&A-intern werk. De domeinvergelijkingen bevatten alleen features voor domeinteams; ontbrekende tijdmetingen worden niet meegerekend. De grafieken tonen uitsluitend geaggregeerde inzichten, geen featuretitels of ruwe registraties.

In [11]:
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA_DIR = Path("../azure_devops_data")
CHARTS_DIR = Path("../docs/assets/charts")
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Gegevens laden en afbakenen

We gebruiken de nieuwste features-analyse-export. De periode start op 20 februari 2025, de vorige retro; de einddatum volgt de laatste sluitdatum in de export.

In [12]:
import re


def _extract_date(path):
    match = re.search(r"(\d{8})", path.stem)
    return match.group(1) if match else ""


xlsx_files = sorted(DATA_DIR.glob("features_analysis_*.xlsx"), key=_extract_date)
if not xlsx_files:
    raise FileNotFoundError(f"No features_analysis_*.xlsx found in {DATA_DIR.resolve()}")

latest_file = xlsx_files[-1]
print(f"Using: {latest_file.name}")

df = pd.read_excel(latest_file)
df["Aangemaakt op"] = pd.to_datetime(
    df["Aangemaakt op"], format="ISO8601", utc=True
).dt.tz_localize(None)
df["Gesloten op"] = pd.to_datetime(df["Gesloten op"], utc=True).dt.tz_localize(None)

PERIOD_START = pd.Timestamp("2025-02-20")
df = df.loc[df["Gesloten op"].notna() & df["Gesloten op"].ge(PERIOD_START)].copy()
PERIOD_END = df["Gesloten op"].max().normalize()
print(f"Afgeronde features van {PERIOD_START:%d-%m-%Y} t/m {PERIOD_END:%d-%m-%Y}: {len(df)}")
df.shape

Using: features_analysis_20261002.xlsx
Afgeronde features van 20-02-2025 t/m 28-09-2026: 126


(126, 15)

## 2. Scope en visuele stijl

`is_DA` bepaalt of een feature D&A-intern is. De eerste grafiek toont alle afgeronde features, met D&A-intern als aparte groep en domeinteams uitgesplitst naar `Area`. De twee doorlooptijdgrafieken tonen alleen domeinteams.

We vergelijken gemiddelde én mediane dagen. Het aantal beschikbare metingen staat in de hoverinformatie; missende metingen tellen dus niet als nul.

In [22]:
import plotly.graph_objects as go
import plotly.io as pio

# Palette follows docs/assets/style.css (hero gradient: navy -> blue -> teal).
NAVY = "#15335b"
DOMAIN_COLOR = "#1d4e89"
INTERNAL_COLOR = "#0f766e"
ACCENT_COLOR = "#f59e0b"
GRID_COLOR = "#e8edf3"
TEXT_COLOR = "#1f2a37"
METRIC_COLORS = {"Gemiddelde": DOMAIN_COLOR, "Mediaan": ACCENT_COLOR}

_axis = {
    "gridcolor": GRID_COLOR,
    "zeroline": False,
    "showline": False,
    "ticks": "",
    "tickfont": {"size": 12},
    "title": {"font": {"size": 12, "color": "#58677a"}},
}
pio.templates["hu"] = go.layout.Template(
    layout={
        "colorway": [DOMAIN_COLOR, INTERNAL_COLOR, ACCENT_COLOR, "#6d8fc0"],
        "xaxis": _axis,
        "yaxis": {**_axis, "gridcolor": "rgba(0,0,0,0)"},
        "title": {"font": {"size": 19, "color": NAVY}, "x": 0.01, "xanchor": "left"},
        "hoverlabel": {
            "bgcolor": NAVY,
            "font": {"family": "Segoe UI, sans-serif", "color": "white"},
            "bordercolor": NAVY,
        },
        "legend": {"orientation": "h", "y": 1.02, "yanchor": "bottom", "x": 1, "xanchor": "right"},
    },
    data={"bar": [{"marker": {"cornerradius": 6, "line": {"width": 0}}}]},
)
TEMPLATE = "plotly_white+hu"
CHART_STYLE = {
    "font": {"family": "Segoe UI, sans-serif", "color": TEXT_COLOR},
    "paper_bgcolor": "rgba(0,0,0,0)",
    "plot_bgcolor": "rgba(0,0,0,0)",
    "margin": {"l": 30, "r": 40, "t": 90, "b": 40},
}

domain_features = df.loc[~df["is_DA"].fillna(False)].copy()
domain_features["Area"] = domain_features["Area"].fillna("Onbekend")


## 3. Het verhaal in drie grafieken

1. **Wat is afgerond?** Totaalbeeld, D&A-intern apart en aantallen per domeinteam.
2. **Hoe lang stond het te wachten?** Gemiddelde en mediaan wachttijd per domeinteam.
3. **Hoe lang was het onderhanden?** Gemiddelde en mediaan doorlooptijd per domeinteam.

De laatste twee grafieken gebruiken alleen beschikbare tijdmetingen van afgeronde domeinteams-features; hover voor het aantal metingen.

In [23]:
# Chart 1: all closed features, split into domain teams and D&A-internal work.
area_counts = (
    domain_features.groupby("Area", dropna=False)
    .size()
    .rename("Aantal features")
    .reset_index()
    .rename(columns={"Area": "Onderdeel"})
)
internal_count = int(df["is_DA"].fillna(False).sum())
total_count = len(df)
completion = pd.concat(
    [
        area_counts.assign(Type="Domeinteams"),
        pd.DataFrame(
            {"Onderdeel": ["D&A intern"], "Aantal features": [internal_count], "Type": ["D&A intern"]}
        ),
    ],
    ignore_index=True,
)
completion["Percentage van totaal"] = completion["Aantal features"].div(total_count).mul(100)
completion_order = completion.sort_values("Aantal features")["Onderdeel"].tolist()

domain_count = len(domain_features)
fig_completed = px.bar(
    completion,
    x="Aantal features",
    y="Onderdeel",
    color="Type",
    orientation="h",
    text="Aantal features",
    custom_data=["Percentage van totaal"],
    category_orders={"Onderdeel": completion_order},
    color_discrete_map={"Domeinteams": DOMAIN_COLOR, "D&A intern": INTERNAL_COLOR},
    template=TEMPLATE,
    title=(
        f"{total_count} features afgerond · {domain_count} voor domeinteams · "
        f"{internal_count} D&A-intern"
    ),
)
fig_completed.update_traces(
    textposition="outside",
    cliponaxis=False,
    hovertemplate="%{y}<br>%{customdata[0]:.1f}% van alle afgeronde features<extra></extra>",
)
fig_completed.update_layout(
    **CHART_STYLE,
    height=max(520, len(completion_order) * 34 + 150),
    xaxis_title="Aantal afgeronde features",
    yaxis_title="",
    legend_title_text="Werkstroom",
    showlegend=True,
)
fig_completed.show()

In [24]:
# Chart 2: average and median waiting time for domain-team features.
waiting_column = "Wachttijd (dagen)"
waiting_stats = (
    domain_features.dropna(subset=[waiting_column])
    .groupby("Area")[waiting_column]
    .agg(["count", "mean"])
    .rename(columns={"count": "Aantal features", "mean": "Gemiddelde"})
)
waiting_order = waiting_stats.sort_values("Gemiddelde").index.tolist()
waiting_long = waiting_stats.reset_index().melt(
    id_vars=["Area", "Aantal features"],
    value_vars=["Gemiddelde"],
    var_name="Maatstaf",
    value_name="Dagen",
)

fig_waiting = px.scatter(
    waiting_long,
    x="Dagen",
    y="Area",
    color="Maatstaf",
    symbol="Maatstaf",
    category_orders={"Area": waiting_order, "Maatstaf": ["Gemiddelde"]},
    color_discrete_map=METRIC_COLORS,
    hover_data={"Area": False, "Maatstaf": True, "Aantal features": True, "Dagen": ":.1f"},
    template=TEMPLATE,
    title="Wachttijd van features voor domeinteams (geprioriteerd → in progress)",
)
fig_waiting.update_traces(marker={"size": 12})
fig_waiting.update_layout(
    **CHART_STYLE,
    height=max(440, len(waiting_order) * 42 + 150),
    xaxis_title="Wachttijd (kalenderdagen)",
    yaxis_title="",
    legend_title_text="Maatstaf",
    hovermode="closest",
)
fig_waiting.show()

In [25]:
# Chart 3: work-in-progress duration for domain-team features.
wip_column = "Onderhanden werk (dagen)"
wip_stats = (
    domain_features.dropna(subset=[wip_column])
    .groupby("Area")[wip_column]
    .agg(["count", "mean"])
    .rename(columns={"count": "Aantal features", "mean": "Gemiddelde"})
)
wip_order = wip_stats.sort_values("Gemiddelde").index.tolist()
wip_long = wip_stats.reset_index().melt(
    id_vars=["Area", "Aantal features"],
    value_vars=["Gemiddelde"],
    var_name="Maatstaf",
    value_name="Dagen",
)

fig_wip = px.scatter(
    wip_long,
    x="Dagen",
    y="Area",
    color="Maatstaf",
    symbol="Maatstaf",
    category_orders={"Area": wip_order, "Maatstaf": ["Gemiddelde"]},
    color_discrete_map=METRIC_COLORS,
    hover_data={"Area": False, "Maatstaf": True, "Aantal features": True, "Dagen": ":.1f"},
    template=TEMPLATE,
    title="Van startsein tot finish: hoe lang liep het werk?",
)
fig_wip.update_traces(marker={"size": 12})
fig_wip.update_layout(
    **CHART_STYLE,
    height=max(440, len(wip_order) * 42 + 150),
    xaxis_title="Onderhanden werk (kalenderdagen): In progress → Done",
    yaxis_title="",
    legend_title_text="Maatstaf",
    hovermode="closest",
)
fig_wip.show()

In [8]:
domain_features.head()

,Feature id,Feature naam,Aangemaakt op,Gesloten op,Effort,Area,Business value,Aantal user stories,Epic id,Epic naam,is_DA,Wachttijd (dagen),Onderhanden werk (dagen),Jaar-Maand,summed_closed_effort_per_year_month
26,107700,ADF: Ontsluiting Netive VMS ( PNIL Registratie),2024-11-14 14:49:06.370,2026-08-20,NaN,Personeel,NaN,1.0,99757,D&A: ADF productie inrichten (2024-2025),False,NaN,NaN,2026-08,2
31,123712,Priva sensordata naar ADF ODS ACC voor BV,2025-05-20 14:14:45.297,2026-07-16,5.0,Bedrijfsvoering,13.0,9.0,108060,Ruimtegebruik,False,97.0,573.0,2026-07,13
32,109826,Overname DM Roostering (DD BV),2024-12-10 11:36:37.140,2025-07-11,NaN,Bedrijfsvoering,NaN,2.0,108060,Ruimtegebruik,False,NaN,16.0,2025-07,1
33,109820,Views op DM en ODS tabellen,2024-12-10 10:54:39.047,2025-03-31,NaN,Bedrijfsvoering,NaN,1.0,108060,Ruimtegebruik,False,NaN,NaN,2025-03,4
34,128895,Prototype OnStage data naar Productie-status,2025-07-18 06:26:48.050,2026-03-16,5.0,Bedrijfsvoering,20.0,9.0,108063,Processen Onderwijslogistiek,False,0.0,118.0,2026-03,30


In [29]:
# Chart 4: closed effort per month for domain-team features.
effort_value_column = "Gesloten effort per jaar-maand"

# Rebuild the month + effort fields from the data actually present in this notebook.
if effort_value_column not in domain_features.columns:
    effort_value_column = "Effort"

if "Jaar-Maand" not in domain_features.columns:
    domain_features["Jaar-Maand"] = pd.to_datetime(domain_features["Gesloten op"]).dt.to_period("M").dt.to_timestamp()

effort_by_month = (
    domain_features.loc[:, ["Jaar-Maand", effort_value_column]]
    .dropna(subset=["Jaar-Maand", effort_value_column])
    .groupby("Jaar-Maand", as_index=False)[effort_value_column]
    .sum()
    .sort_values("Jaar-Maand")
    .reset_index(drop=True)
)

fig_effort = px.bar(
    effort_by_month,
    x="Jaar-Maand",
    y=effort_value_column,
    text=effort_value_column,
    template=TEMPLATE,
    title="Effort per maand",
)
fig_effort.update_traces(
    texttemplate="%{y:.0f}",
    textposition="outside",
    hovertemplate="%{x}<br>Effort: %{y:.0f}<extra></extra>",
)
fig_effort.update_layout(
    **CHART_STYLE,
    height=480,
    xaxis_title="Maand",
    yaxis_title="Effort",
    showlegend=False,
)
fig_effort.show()

In [27]:
# Chart 5: summed closed effort per domain team (excl. D&A-internal).
effort_by_area = (
    domain_features.dropna(subset=["Effort"])
    .groupby("Area", as_index=False)["Effort"]
    .sum()
    .sort_values("Effort")
)
effort_area_order = effort_by_area["Area"].tolist()
effort_area_mean = effort_by_area["Effort"].mean()

fig_effort_area = px.bar(
    effort_by_area,
    x="Effort",
    y="Area",
    orientation="h",
    text="Effort",
    category_orders={"Area": effort_area_order},
    color_discrete_sequence=[DOMAIN_COLOR],
    template=TEMPLATE,
    title="Totale effort van afgeronde features per domeinteam",
)
fig_effort_area.update_traces(
    texttemplate="%{x:.0f}",
    textposition="outside",
    cliponaxis=False,
    hovertemplate="%{y}<br>Totale effort: %{x:.0f}<extra></extra>",
)
fig_effort_area.add_vline(
    x=effort_area_mean,
    line_dash="dash",
    line_color="#f59e0b",
    annotation_text=f"Gemiddeld: {effort_area_mean:.1f}",
    annotation_position="top",
)
fig_effort_area.update_layout(
    **CHART_STYLE,
    height=max(440, len(effort_area_order) * 34 + 150),
    xaxis_title="Totale effort (afgeronde features)",
    yaxis_title="",
    showlegend=False,
)
fig_effort_area.show()


## 4. Publiceren op GitHub Pages

Exporteer alleen de drie geaggregeerde grafieken met vaste bestandsnamen; `docs/index.html` linkt hiernaar. Featuretitels, ID's en ruwe registraties worden niet gepubliceerd.

In [30]:
charts = {
    "features_completed.html": fig_completed,
    "waiting_time_by_area.html": fig_waiting,
    "wip_duration_by_area.html": fig_wip,
    "closed_effort_per_month.html": fig_effort,
    "closed_effort_by_area.html": fig_effort_area,
}

for filename, fig in charts.items():
    fig.write_html(CHARTS_DIR / filename, include_plotlyjs="cdn", full_html=True)

print(f"Exported {len(charts)} charts to {CHARTS_DIR.resolve()}")
print(f"Source file: {latest_file.name}")


Exported 5 charts to C:\Users\fraukje.coopmans\OneDrive - Stichting Hogeschool Utrecht\Documents\git repos\dena_KPI_Dashboard\docs\assets\charts
Source file: features_analysis_20261002.xlsx
